# Kubeflow Pipeline

A minimal pipeline, to confirm the notebook can compile and submit before
building the real one.

References:

- <https://www.kubeflow.org/docs/components/pipelines/getting-started/>
- <https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/connect-api/>

## Environment

In [ ]:
# pip install
%pip install -q -U kfp

## Define

A component is a type-annotated Python function.

In [ ]:
from kfp import dsl


@dsl.component
def say_hello(name: str) -> str:
    hello_text = f"Hello, {name}!"
    print(hello_text)
    return hello_text


@dsl.pipeline
def hello_pipeline(recipient: str) -> str:
    hello_task = say_hello(name=recipient)
    return hello_task.output

## Compile

Produces a self-contained pipeline yaml. Needs no cluster.

In [ ]:
from kfp import compiler

compiler.Compiler().compile(hello_pipeline, "pipeline.yaml")

## Connect

Inside the cluster `kfp.Client()` needs no arguments: it reads the token from
`KF_PIPELINES_SA_TOKEN_PATH` and defaults to
`http://ml-pipeline-ui.kubeflow.svc.cluster.local`.

The token volume comes from a `PodDefault` in this profile namespace.

In [ ]:
import kfp

kfp_client = kfp.Client()

# test the client by listing experiments
experiments = kfp_client.list_experiments(namespace="kubeflow-user-example-com")
print(experiments)

## Run

In [ ]:
run = kfp_client.create_run_from_pipeline_package(
    "pipeline.yaml",
    arguments={
        "recipient": "World",
    },
)

print(run.run_id)